# Machine Learning Modeling
This notebook builds and evaluates a Random Forest classifier to predict pain medication effectiveness.

## 1. Import Required Libraries

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

# Scikit-learn imports
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, 
    confusion_matrix, 
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# Visualization settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

## 2. Load Processed Data

In [ ]:
# TODO: Load ML-ready data
processed_data_path = '../../data/processed/pain_meds_ml_ready.csv'

df = pd.read_csv(processed_data_path)

print(f"Processed data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nColumns preview:")
print(df.columns.tolist()[:20])  # Show first 20 columns
display(df.head())

## 3. Split into Features (X) and Target (y)

In [ ]:
# TODO: Prepare features and target variable
# Define target variable
target_col = 'effectiveness_encoded'

# Define columns to exclude from features
exclude_cols = [
    'effectiveness', 'effectiveness_encoded', 'rating',
    'drugName', 'condition', 'drugName_top', 'condition_top', 
    'review', 'date'  # Add any other non-feature columns
]

# Get feature columns
feature_cols = [col for col in df.columns if col not in exclude_cols]

# Create X (features) and y (target)
X = df[feature_cols]
y = df[target_col]

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print(f"\nNumber of features: {len(feature_cols)}")
print(f"\nTarget distribution:")
print(y.value_counts().sort_index())
print("\nTarget class percentages:")
print(y.value_counts(normalize=True).sort_index() * 100)

## 4. Train/Test Split (80/20, Stratified)

In [ ]:
# TODO: Split data into train and test sets
# Use stratified split to maintain class distribution

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print("Data split complete!")
print("=" * 60)
print(f"Training set size: {len(X_train):,} ({(len(X_train)/len(X))*100:.1f}%)")
print(f"Test set size: {len(X_test):,} ({(len(X_test)/len(X))*100:.1f}%)")
print("\nTraining set target distribution:")
print(y_train.value_counts(normalize=True).sort_index() * 100)
print("\nTest set target distribution:")
print(y_test.value_counts(normalize=True).sort_index() * 100)

## 5. Train Random Forest Classifier

In [ ]:
# TODO: Train Random Forest model
print("Training Random Forest Classifier...")

# Initialize the model
rf_model = RandomForestClassifier(
    n_estimators=100,      # Number of trees
    max_depth=20,          # Maximum depth of trees
    min_samples_split=10,  # Minimum samples to split a node
    min_samples_leaf=5,    # Minimum samples at leaf node
    random_state=42,
    n_jobs=-1,             # Use all processors
    verbose=1
)

# Train the model
rf_model.fit(X_train, y_train)

print("\nModel training complete!")
print(f"Number of trees: {rf_model.n_estimators}")
print(f"Number of features used: {rf_model.n_features_in_}")
print(f"Number of classes: {len(rf_model.classes_)}")

## 6. Make Predictions and Get Probabilities

In [ ]:
# TODO: Generate predictions on test set
print("Generating predictions...")

# Predictions on training set
y_train_pred = rf_model.predict(X_train)
y_train_proba = rf_model.predict_proba(X_train)

# Predictions on test set
y_test_pred = rf_model.predict(X_test)
y_test_proba = rf_model.predict_proba(X_test)

print("Predictions generated!")
print(f"\nTraining set predictions shape: {y_train_pred.shape}")
print(f"Test set predictions shape: {y_test_pred.shape}")
print(f"Probability matrix shape: {y_test_proba.shape}")

# Display sample predictions with probabilities
print("\nSample predictions:")
sample_results = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': y_test_pred[:10],
    'Prob_Class_0': y_test_proba[:10, 0],
    'Prob_Class_1': y_test_proba[:10, 1],
    'Prob_Class_2': y_test_proba[:10, 2]
})
display(sample_results)

## 7. Calculate Metrics (Accuracy, Confusion Matrix, Classification Report)

In [ ]:
# TODO: Calculate and display model performance metrics
print("=" * 60)
print("MODEL PERFORMANCE METRICS")
print("=" * 60)

# Training accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
print(f"\nTraining Accuracy: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")

# Test accuracy
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# Additional metrics
test_precision = precision_score(y_test, y_test_pred, average='weighted')
test_recall = recall_score(y_test, y_test_pred, average='weighted')
test_f1 = f1_score(y_test, y_test_pred, average='weighted')

print(f"\nTest Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test F1-Score: {test_f1:.4f}")

# Confusion Matrix
print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)
cm = confusion_matrix(y_test, y_test_pred)
print(cm)

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Not Effective', 'Partially Effective', 'Effective'],
            yticklabels=['Not Effective', 'Partially Effective', 'Effective'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix - Pain Medication Effectiveness Prediction')
plt.tight_layout()
plt.show()

# Classification Report
print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
class_names = ['Not Effective (0)', 'Partially Effective (1)', 'Effective (2)']
print(classification_report(y_test, y_test_pred, target_names=class_names))

## 8. Feature Importance Analysis

In [ ]:
# TODO: Analyze feature importance
print("Analyzing feature importance...")

# Get feature importances
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

# Display top 20 features
print("\nTop 20 Most Important Features:")
print("=" * 60)
display(feature_importance.head(20))

# Visualize top 15 features
plt.figure(figsize=(12, 8))
top_15 = feature_importance.head(15)
plt.barh(range(len(top_15)), top_15['importance'].values, color='steelblue')
plt.yticks(range(len(top_15)), top_15['feature'].values)
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Top 15 Feature Importances - Random Forest Model', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Summary statistics
print(f"\nTotal features: {len(feature_importance)}")
print(f"Features with importance > 0.01: {len(feature_importance[feature_importance['importance'] > 0.01])}")
print(f"Features with importance > 0.05: {len(feature_importance[feature_importance['importance'] > 0.05])}")

## 9. Save Model

In [ ]:
# TODO: Save trained model to file
model_output_path = '../../outputs/models/rf_model.pkl'

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(model_output_path), exist_ok=True)

# Save model using pickle
with open(model_output_path, 'wb') as f:
    pickle.dump(rf_model, f)

print(f"Model saved to: {model_output_path}")

# Also save feature names for later use
feature_names_path = '../../outputs/models/feature_names.pkl'
with open(feature_names_path, 'wb') as f:
    pickle.dump(feature_cols, f)

print(f"Feature names saved to: {feature_names_path}")

# Save feature importance
feature_importance_path = '../../outputs/models/feature_importance.csv'
feature_importance.to_csv(feature_importance_path, index=False)
print(f"Feature importance saved to: {feature_importance_path}")

# Save test predictions for analysis
test_results_path = '../../outputs/models/test_predictions.csv'
test_results = pd.DataFrame({
    'actual': y_test.values,
    'predicted': y_test_pred,
    'prob_not_effective': y_test_proba[:, 0],
    'prob_partially_effective': y_test_proba[:, 1],
    'prob_effective': y_test_proba[:, 2]
})
test_results.to_csv(test_results_path, index=False)
print(f"Test predictions saved to: {test_results_path}")

print("\n" + "=" * 60)
print("MODELING COMPLETE")
print("=" * 60)
print(f"Final Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Model and results saved successfully!")
print("=" * 60)